<a href="https://colab.research.google.com/github/AlexMora15/tecnm-desarrollo-web/blob/main/laboratorios/sesion03/sesion03_laboratorio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Desarrollo de Aplicaciones Web (DAB-2201)
## Sesión 03: Headers, Content-Type y Negociación de Contenido
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-desarrollo-web`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *Lectura y emisión manual de encabezados HTTP*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Invariantes de negociación (Content-Type vs Accept)* y formular prompt.
3. **Fase 3 (El Auditor):** Someter el servidor al Test Harness: *Pruebas con tipos de contenido no soportados (415)*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion03.md`.

### Paso 0: Inicialización del Entorno (FastAPI y PHP 8+)

In [13]:
import os
print('⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...')
!pip install -q fastapi uvicorn httpx pydantic passlib python-jose[cryptography] sqlalchemy alembic
if os.system('which php > /dev/null 2>&1') != 0:
    print('⏳ Instalando PHP CLI...')
    !apt-get update -qq && apt-get install -y -qq php-cli > /dev/null 2>&1
print('✅ Entorno Web listo:')
!python3 --version
!php -v | head -n 1

⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...
✅ Entorno Web listo:
Python 3.13.18
PHP 8.3.6 (cli) (built: Oct 19 2026 15:16:58) (NTS)


---
## SECCIÓN A: TRACK PYTHON / FASTAPI
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Lectura y emisión manual de encabezados HTTP

In [ ]:
%%writefile app_artesano.py
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel

app = FastAPI(title="API Artesano")

class PayloadRequest(BaseModel):
    dato: str

# TODO (El Artesano): Implementar lógica manual del endpoint
@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):
    if not payload.dato.strip():
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Dato vacio")
    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [16]:
%%writefile app_ia.py
from fastapi import FastAPI, Request, HTTPException, status, Response
from pydantic import BaseModel
import json

app = FastAPI(title="API Cientifico Optimizado")

class PayloadRequest(BaseModel):
    dato: str

@app.post("/api/recurso")
async def procesar_recurso_optimizado(request: Request):
    # 1. Validar estrictamente el Content-Type
    content_type = request.headers.get("content-type", "")
    if "application/json" not in content_type:
        raise HTTPException(
            status_code=status.HTTP_415_UNSUPPORTED_MEDIA_TYPE,
            detail=f"Content-Type '{content_type}' no soportado. Debe ser 'application/json'."
        )

    # 2. Parsear y validar el cuerpo de la petición
    try:
        body_bytes = await request.body()
        body_data = json.loads(body_bytes)
        payload = PayloadRequest(**body_data)
    except Exception:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Estructura o formato JSON invalido"
        )

    # 3. Validar contenido del negocio
    if not payload.dato.strip():
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Dato vacio"
        )

    # 4. Construir respuesta con encabezados de negociación exactos
    headers = {
        "Content-Type": "application/json",
        "X-Content-Negotiation": "Verified",
        "Cache-Control": "no-store"
    }

    contenido_respuesta = {
        "status": "success",
        "mensaje": f"Procesado: {payload.dato}"
    }

    return Response(
        content=json.dumps(contenido_respuesta),
        media_type="application/json",
        headers=headers,
        status_code=status.HTTP_200_OK
    )

Overwriting app_ia.py


In [14]:
%%writefile app_ia.py
from fastapi import FastAPI, Request, HTTPException, status, Response
from pydantic import BaseModel
import json

app = FastAPI(title="API Cientifico Optimizado")

class PayloadRequest(BaseModel):
    dato: str

@app.post("/api/recurso")
async def procesar_recurso_optimizado(request: Request):
    # 1. Validar estrictamente el Content-Type
    content_type = request.headers.get("content-type", "")
    if "application/json" not in content_type:
        raise HTTPException(
            status_code=status.HTTP_415_UNSUPPORTED_MEDIA_TYPE,
            detail=f"Content-Type '{content_type}' no soportado. Debe ser 'application/json'."
        )

    # 2. Parsear y validar el cuerpo de la petición
    try:
        body_bytes = await request.body()
        body_data = json.loads(body_bytes)
        payload = PayloadRequest(**body_data)
    except Exception:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Estructura o formato JSON invalido"
        )

    # 3. Validar contenido del negocio
    if not payload.dato.strip():
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Dato vacio"
        )

    # 4. Construir respuesta con encabezados de negociación exactos
    headers = {
        "Content-Type": "application/json",
        "X-Content-Negotiation": "Verified",
        "Cache-Control": "no-store"
    }

    contenido_respuesta = {
        "status": "success",
        "mensaje": f"Procesado: {payload.dato}"
    }

    return Response(
        content=json.dumps(contenido_respuesta),
        media_type="application/json",
        headers=headers,
        status_code=status.HTTP_200_OK
    )

Overwriting app_ia.py


### Fase 3: El Auditor (Test Harness HTTP)
Pruebas de servidor: Pruebas con tipos de contenido no soportados (415)

In [20]:
import threading
import time
import uvicorn
import httpx
import socket
import sys
import importlib

# Asegurar que el archivo escrito en el disco sea el que Python cargue como módulo
if 'app_ia' in sys.modules:
    del sys.modules['app_ia']
import app_ia
from app_ia import app

def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('127.0.0.1', 0))
        return s.getsockname()[1]

PORT = find_free_port()

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)

client = httpx.Client(base_url=f"http://127.0.0.1:{PORT}")
print("=== AUDITORÍA CRÍTICA WEB: SESIÓN 03 ===\n")

# Caso 1: Petición correcta (Content-Type: application/json)
print("[Prueba 1] Enviando JSON válido...")
r1 = client.post("/api/recurso", json={"dato": "Prueba Test Harness"})
print(f"Respuesta 1: HTTP {r1.status_code}")
print(f"Headers: {dict(r1.headers)}")
print(f"Cuerpo: {r1.json()}")
assert r1.status_code == 200

# Verificación robusta e insensible a mayúsculas/minúsculas de las llaves del encabezado
headers_lower = {k.lower(): v for k, v in r1.headers.items()}
assert headers_lower.get("x-content-negotiation") == "verified" or headers_lower.get("x-content-negotiation") == "Verified" or headers_lower.get("x-content-negotiated") == "application/json"
print("PASO 1 ✅ - Negociación de contenido correcta y headers validados.\n")

# Caso 2: Petición con Content-Type no soportado (Provocar 415)
print("[Prueba 2] Enviando tipo de contenido inválido (text/plain)...")
r2 = client.post(
    "/api/recurso",
    content="dato=Prueba",
    headers={"Content-Type": "text/plain"}
)
print(f"Respuesta 2: HTTP {r2.status_code}")
print(f"Cuerpo: {r2.text}")
assert r2.status_code == 415
print("PASO 2 ✅ - El servidor rechazó correctamente el Content-Type no admitido con HTTP 415.\n")

print("🚀 ¡TODAS LAS PRUEBAS DE AUDITORÍA PASARON CON ÉXITO!")

=== AUDITORÍA CRÍTICA WEB: SESIÓN 03 ===

[Prueba 1] Enviando JSON válido...
Respuesta 1: HTTP 200
Headers: {'date': 'Thu, 19 Nov 2026 16:12:40 GMT', 'server': 'uvicorn', 'content-type': 'application/json', 'x-content-negotiation': 'Verified', 'cache-control': 'no-store', 'content-length': '87'}
Cuerpo: {'status': 'success', 'mensaje': 'Procesado de manera cientifica: Prueba Test Harness'}
PASO 1 ✅ - Negociación de contenido correcta y headers validados.

[Prueba 2] Enviando tipo de contenido inválido (text/plain)...
Respuesta 2: HTTP 415
Cuerpo: {"detail":"Content-Type 'text/plain' no soportado. Debe ser 'application/json'."}
PASO 2 ✅ - El servidor rechazó correctamente el Content-Type no admitido con HTTP 415.

🚀 ¡TODAS LAS PRUEBAS DE AUDITORÍA PASARON CON ÉXITO!
